In [32]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [33]:
df = pd.read_csv("03_aspi_merged.csv")

# Convert to correct types
df['date'] = pd.to_datetime(df['date'])

# Important numeric columns
df['close'] = pd.to_numeric(df['close'], errors='coerce')
df['aspi_close'] = pd.to_numeric(df['aspi_close'], errors='coerce')

# Sort properly
df = df.sort_values(['symbol', 'date'])

In [34]:
df.groupby('symbol')[['symbol', 'date']].head()

,symbol,date
4638,AAF,2025-02-03
4895,AAF,2025-02-03
4639,AAF,2025-02-05
4896,AAF,2025-02-05
4640,AAF,2025-02-06
...,...,...
49016,WAPO,2025-02-03
49017,WAPO,2025-02-05
49018,WAPO,2025-02-06
49019,WAPO,2025-02-07


In [35]:
# Stock return
df['R_i'] = df.groupby('symbol')['close'].pct_change()

# Market return (same for all stocks per day)
df['R_m'] = df.groupby('symbol')['aspi_close'].pct_change()

In [36]:
event_date = pd.to_datetime("2025-11-28")

df['day'] = (df['date'] - event_date).dt.days

# Estimation window
estimation_df = df[(df['day'] >= -120) & (df['day'] <= -6)]

In [37]:
results = []

for symbol, group in estimation_df.groupby('symbol'):

    # Drop NaNs
    group = group[['R_i', 'R_m']].dropna()

    if len(group) < 30:  # safety check
        continue

    X = group['R_m']
    y = group['R_i']

    # Add constant (for alpha)
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    alpha = model.params['const']
    beta = model.params['R_m']

    results.append({
        'symbol': symbol,
        'alpha': alpha,
        'beta': beta
    })

alpha_beta_df = pd.DataFrame(results)

In [38]:
alpha_beta_df.head()

,symbol,alpha,beta
0,AAF,0.008549,-1.530271
1,AAIC,-0.002666,0.654950
2,ABAN,0.008363,1.633533
3,ABL,0.002495,0.584640
4,ACAP,0.003096,1.401049


In [13]:
alpha_beta_df.to_csv('regression_alpha_beta.csv')

In [41]:
df = df.merge(alpha_beta_df, on='symbol', how='left')
df.head()

,symbol,date,open,high,low,close,volume,suffix,sector,suffix_cat,...,aspi_high,aspi_low,aspi_close,R_i,R_m,day,alpha_x,beta_x,alpha_y,beta_y
0,AAF,2025-02-03,29.3,29.3,28.5,28.8,148508.0,N0000,Finance/Rental/Leasing,0,...,17127.65,16900.36,16956.49,NaN,NaN,-298,0.008549,-1.530271,0.008549,-1.530271
1,AAF,2025-02-03,28.7,28.7,28.6,28.6,1921.0,P0000,Finance/Rental/Leasing,2,...,17127.65,16900.36,16956.49,-0.006944,0.000000,-298,0.008549,-1.530271,0.008549,-1.530271
2,AAF,2025-02-05,28.8,28.9,27.8,27.9,155337.0,N0000,Finance/Rental/Leasing,0,...,16974.58,16440.74,16456.10,-0.024476,-0.029510,-296,0.008549,-1.530271,0.008549,-1.530271
3,AAF,2025-02-05,28.6,29.7,26.6,29.5,5737.0,P0000,Finance/Rental/Leasing,2,...,16974.58,16440.74,16456.10,0.057348,0.000000,-296,0.008549,-1.530271,0.008549,-1.530271
4,AAF,2025-02-06,27.8,29.5,27.5,29.5,166488.0,N0000,Finance/Rental/Leasing,0,...,16675.80,16295.31,16506.73,0.000000,0.003077,-295,0.008549,-1.530271,0.008549,-1.530271


In [40]:
df.to_csv('04_alpha_beta_by_Reg.csv')